In [1]:
import pandas as pd 
df = pd.read_csv('LIAR_bal.csv') 

from sklearn.model_selection import train_test_split 
X_train_text, X_test_text, y_train, y_test = train_test_split( df['title'], df['label'], test_size=0.2, random_state=42 )

In [3]:
# =========================
# Fine-tune RoBERTa (hamzab model)
# =========================

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Dataset Class ----------
class NewsDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load RoBERTa tokenizer & model ----------

MODEL_NAME = "hamzab/roberta-fake-news-classification"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

model.to(device)

# ---------- Datasets ----------

train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

# ---------- Optimizer & Scheduler ----------

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

epochs = 4

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for batch in tqdm(
        train_loader,
        desc=f"Training Epoch {epoch+1}"
    ):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss/len(train_loader):.4f}"
    )

# ---------- Evaluation ----------

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Evaluating"
    ):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# ---------- Metrics ----------

acc = accuracy_score(
    all_labels,
    all_preds
)

f1 = f1_score(
    all_labels,
    all_preds,
    average='weighted'
)

cm = confusion_matrix(
    all_labels,
    all_preds
)

print(
    f"\nValidation Accuracy: {acc:.4f}"
    f" | F1-score: {f1:.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds
    )
)

print("\nConfusion Matrix:")
print(cm)

Training Epoch 1:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.7124


Training Epoch 2:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.6680


Training Epoch 3:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 3 | Train Loss: 0.6490


Training Epoch 4:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 4 | Train Loss: 0.6147


Evaluating:   0%|          | 0/139 [00:00<?, ?it/s]


Validation Accuracy: 0.6416 | F1-score: 0.6410

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.60      0.63      1129
           1       0.62      0.69      0.65      1089

    accuracy                           0.64      2218
   macro avg       0.64      0.64      0.64      2218
weighted avg       0.64      0.64      0.64      2218


Confusion Matrix:
[[677 452]
 [343 746]]


In [1]:
import transformers
print(transformers.__version__)

4.51.3


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Model ----------
MODEL_NAME = "hamzab/roberta-fake-news-classification"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()   # IMPORTANT

# ---------- Dataset ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Data ----------
test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Evaluation ONLY ----------
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")
cm = confusion_matrix(all_labels, all_preds)

print(f"\nAccuracy: {acc:.4f} | F1: {f1:.4f}")
print("\nClassification Report:\n", classification_report(all_labels, all_preds))
print("\nConfusion Matrix:\n", cm)

Evaluating:   0%|          | 0/139 [00:00<?, ?it/s]


Accuracy: 0.4522 | F1: 0.4507

Classification Report:
               precision    recall  f1-score   support

           0       0.46      0.40      0.43      1129
           1       0.45      0.51      0.48      1089

    accuracy                           0.45      2218
   macro avg       0.45      0.45      0.45      2218
weighted avg       0.45      0.45      0.45      2218


Confusion Matrix:
 [[453 676]
 [539 550]]


In [3]:
# =========================
# Fine-tuning BERT (Fake News Detection)
# Model: jy46604790/Fake-News-Bert-Detect
# =========================

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Model Name ----------
MODEL_NAME = "jy46604790/Fake-News-Bert-Detect"

# ---------- Tokenizer & Model ----------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)

# ---------- Dataset Class ----------
class NewsDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Data ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

# ---------- Scheduler ----------
epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# =========================
# TRAINING LOOP
# =========================
for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

# =========================
# EVALUATION
# =========================
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")
cm = confusion_matrix(all_labels, all_preds)

print(f"\nAccuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Epoch 1:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 1 | Loss: 0.6882


Epoch 2:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.6488


Epoch 3:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 3 | Loss: 0.5900


Epoch 4:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 4 | Loss: 0.4891


Evaluating:   0%|          | 0/139 [00:00<?, ?it/s]


Accuracy: 0.6348
F1 Score: 0.6348

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.65      0.64      1129
           1       0.63      0.62      0.63      1089

    accuracy                           0.63      2218
   macro avg       0.63      0.63      0.63      2218
weighted avg       0.63      0.63      0.63      2218


Confusion Matrix:
[[729 400]
 [410 679]]


In [ ]:
# =========================
# Fine-tune DeBERTa
# =========================

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# ---------- Number of labels ----------
num_labels = len(set(y_train))  # or len(set(df['label']))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model (DeBERTa) ----------
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

cuda


C:\Users\HP\anaconda3\envs\torch_gpu\Lib\site-packages\transformers\convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/555 [00:00<?, ?it/s]